In [1]:
from pathlib import Path
import json
import re
import time
from datetime import datetime, timezone

import pandas as pd
import requests

In [2]:
CHIA_DIR = Path("chia_corpus")
LCT_DIR = Path("lct_corpus")

OUTPUT_DIR = Path("benchmark_datasets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHIA_OUTPUT_FILE = (
    OUTPUT_DIR
    / "chia_ctg_retrieval.csv"
)

LCT_OUTPUT_FILE = (
    OUTPUT_DIR
    / "lct_ctg_retrieval.csv"
)

API_URL = "https://clinicaltrials.gov/api/v2/studies"

MAX_API_ATTEMPTS = 5
RETRY_WAIT_SECONDS = 10
REQUEST_PAUSE_SECONDS = 1

In [3]:
CHIA_FILENAME_PATTERN = re.compile(
    r"^(NCT\d{8})_(inc|exc)\.txt$"
)

txt_files = sorted(
    CHIA_DIR.glob("*.txt")
)

trial_files = {}

for path in txt_files:

    match = CHIA_FILENAME_PATTERN.match(
        path.name
    )

    if match is None:
        raise ValueError(
            f"Unexpected CHIA filename: {path.name}"
        )

    nct_id = match.group(1)
    section = match.group(2)

    trial_files.setdefault(
        nct_id,
        {}
    )

    if section in trial_files[nct_id]:
        raise ValueError(
            f"Duplicate {section} file for {nct_id}"
        )

    trial_files[nct_id][section] = path


incomplete_trials = {
    nct_id: sorted(files.keys())
    for nct_id, files in trial_files.items()
    if set(files.keys()) != {"inc", "exc"}
}

if incomplete_trials:
    raise ValueError(
        "Trials with incomplete CHIA "
        f"inclusion/exclusion files: {incomplete_trials}"
    )


chia_nct_ids = sorted(
    trial_files.keys()
)


print(
    f"CHIA TXT files: {len(txt_files)}"
)

print(
    f"CHIA trials identified: {len(chia_nct_ids)}"
)

print(
    f"Unique NCT IDs: {len(set(chia_nct_ids))}"
)

CHIA TXT files: 2000
CHIA trials identified: 1000
Unique NCT IDs: 1000


In [4]:
LCT_FILENAME_PATTERN = re.compile(
    r"^(NCT\d{8})\.txt$"
)

txt_files = sorted(
    LCT_DIR.rglob("*.txt")
)

lct_nct_ids = []

for path in txt_files:

    match = LCT_FILENAME_PATTERN.match(
        path.name
    )

    if match is None:
        raise ValueError(
            f"Unexpected LCT filename: {path.name}"
        )

    lct_nct_ids.append(
        match.group(1)
    )


if len(lct_nct_ids) != len(set(lct_nct_ids)):

    duplicate_ids = sorted(
        pd.Series(lct_nct_ids)[
            pd.Series(lct_nct_ids).duplicated(
                keep=False
            )
        ].unique()
    )

    raise ValueError(
        f"Duplicate LCT NCT IDs found: {duplicate_ids}"
    )


lct_nct_ids = sorted(
    lct_nct_ids
)


print(
    f"LCT TXT files: {len(txt_files)}"
)

print(
    f"LCT trials identified: {len(lct_nct_ids)}"
)

print(
    f"Unique NCT IDs: {len(set(lct_nct_ids))}"
)

LCT TXT files: 1006
LCT trials identified: 1006
Unique NCT IDs: 1006


In [5]:
def fetch_ctg_study(nct_id):
    """
    Retrieve one ClinicalTrials.gov study by NCT ID.

    If the requested NCT ID has been replaced by a current NCT ID,
    the remapping is accepted only when the requested ID is explicitly
    listed by ClinicalTrials.gov in nctIdAliases.

    Returns
    -------
    study : dict
        Full ClinicalTrials.gov study JSON.
    resolved_nct_id : str
        Current NCT ID returned by ClinicalTrials.gov.
    id_remapped : bool
        Whether the requested NCT ID resolved to a different current NCT ID.
    """

    url = f"{API_URL}/{nct_id}"

    last_error = None

    for attempt in range(
        1,
        MAX_API_ATTEMPTS + 1,
    ):

        try:
            response = requests.get(
                url,
                timeout=60,
            )

            response.raise_for_status()

            study = response.json()

            identification = (
                study
                .get("protocolSection", {})
                .get("identificationModule", {})
            )

            returned_nct_id = identification.get(
                "nctId"
            )

            nct_id_aliases = (
                identification.get(
                    "nctIdAliases",
                    [],
                )
                or []
            )

            if returned_nct_id == nct_id:
                id_remapped = False

            elif nct_id in nct_id_aliases:
                id_remapped = True

            else:
                raise RuntimeError(
                    f"NCT ID mismatch: requested {nct_id}, "
                    f"received {returned_nct_id}, and requested "
                    "ID was not listed in nctIdAliases"
                )

            return (
                study,
                returned_nct_id,
                id_remapped,
            )

        except Exception as exc:
            last_error = exc

            if attempt < MAX_API_ATTEMPTS:
                time.sleep(
                    RETRY_WAIT_SECONDS
                )

    raise RuntimeError(
        f"Failed to retrieve {nct_id} after "
        f"{MAX_API_ATTEMPTS} attempts: {last_error}"
    )

In [6]:
def extract_benchmark_record(
    study,
    dataset_label,
):
    """
    Extract the benchmark analysis fields from a
    ClinicalTrials.gov study record.
    """

    protocol = study.get(
        "protocolSection",
        {},
    )

    identification = protocol.get(
        "identificationModule",
        {},
    )

    eligibility = protocol.get(
        "eligibilityModule",
        {},
    )

    return {
        "dataset": dataset_label,
        "resolved_nct_id": identification.get(
            "nctId"
        ),
        "eligibility_text": eligibility.get(
            "eligibilityCriteria"
        ),
    }

In [8]:
def retrieve_benchmark_dataset(
    nct_ids,
    dataset_label,
    output_file,
):
    """
    Retrieve ClinicalTrials.gov eligibility text for a fixed benchmark
    set of NCT IDs.

    Successful records already present in output_file are skipped.
    Failed records are retried and replaced, ensuring one row per
    benchmark NCT ID.
    """

    expected_columns = [
        "dataset",
        "nct_id",
        "resolved_nct_id",
        "id_remapped",
        "request_success",
        "api_error",
        "eligibility_text",
        "retrieved_at",
    ]

    if output_file.exists():

        existing_df = pd.read_csv(
            output_file,
            encoding="utf-8-sig",
        )

        missing_columns = (
            set(expected_columns)
            - set(existing_df.columns)
        )

        if missing_columns:
            raise RuntimeError(
                f"Incompatible checkpoint file: {output_file}\n"
                f"Missing columns: {sorted(missing_columns)}"
            )

        if existing_df["nct_id"].duplicated().any():
            raise RuntimeError(
                f"Duplicate NCT IDs found in checkpoint: {output_file}"
            )

        records_by_id = {
            row["nct_id"]: row
            for row in existing_df.to_dict(
                orient="records"
            )
        }

    else:

        records_by_id = {}

    total = len(nct_ids)

    for i, nct_id in enumerate(
        nct_ids,
        start=1,
    ):

        existing_record = records_by_id.get(
            nct_id
        )

        if (
            existing_record is not None
            and existing_record["request_success"] == True
        ):
            print(
                f"[{i}/{total}] "
                f"Skipping completed: {nct_id}"
            )
            continue

        print(
            f"[{i}/{total}] "
            f"Retrieving: {nct_id}"
        )

        try:

            (
                study,
                resolved_nct_id,
                id_remapped,
            ) = fetch_ctg_study(
                nct_id
            )

            extracted = extract_benchmark_record(
                study,
                dataset_label,
            )

            record = {
                "dataset": dataset_label,
                "nct_id": nct_id,
                "resolved_nct_id": resolved_nct_id,
                "id_remapped": id_remapped,
                "request_success": True,
                "api_error": None,
                "eligibility_text": extracted[
                    "eligibility_text"
                ],
                "retrieved_at": datetime.now(
                    timezone.utc
                ).isoformat(),
            }

        except Exception as exc:

            record = {
                "dataset": dataset_label,
                "nct_id": nct_id,
                "resolved_nct_id": None,
                "id_remapped": False,
                "request_success": False,
                "api_error": str(exc),
                "eligibility_text": None,
                "retrieved_at": datetime.now(
                    timezone.utc
                ).isoformat(),
            }

        # Replace any previous record for this NCT ID.
        records_by_id[nct_id] = record

        checkpoint_df = pd.DataFrame(
            [
                records_by_id[x]
                for x in nct_ids
                if x in records_by_id
            ],
            columns=expected_columns,
        )

        checkpoint_df.to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig",
        )

        time.sleep(
            REQUEST_PAUSE_SECONDS
        )

    return pd.DataFrame(
        [
            records_by_id[x]
            for x in nct_ids
            if x in records_by_id
        ],
        columns=expected_columns,
    )

In [9]:
test_nct_id = "NCT02822001"

(
    test_study,
    test_resolved_nct_id,
    test_id_remapped,
) = fetch_ctg_study(
    test_nct_id
)

test_record = extract_benchmark_record(
    test_study,
    "CHIA",
)

print(
    "Requested NCT ID:",
    test_nct_id,
)

print(
    "Resolved NCT ID:",
    test_resolved_nct_id,
)

print(
    "ID remapped:",
    test_id_remapped,
)

print(
    "Eligibility text available:",
    bool(
        test_record["eligibility_text"]
    ),
)

print(
    "Eligibility text length:",
    len(
        test_record["eligibility_text"]
        or ""
    ),
)

Requested NCT ID: NCT02822001
Resolved NCT ID: NCT02728726
ID remapped: True
Eligibility text available: True
Eligibility text length: 432


In [10]:
chia_retrieval_df = retrieve_benchmark_dataset(
    nct_ids=chia_nct_ids,
    dataset_label="CHIA",
    output_file=CHIA_OUTPUT_FILE,
)

[1/1000] Retrieving: NCT00050349
[2/1000] Retrieving: NCT00061308
[3/1000] Retrieving: NCT00094861
[4/1000] Retrieving: NCT00122070
[5/1000] Retrieving: NCT00182520
[6/1000] Retrieving: NCT00183885
[7/1000] Retrieving: NCT00198913
[8/1000] Retrieving: NCT00235170
[9/1000] Retrieving: NCT00236340
[10/1000] Retrieving: NCT00250640
[11/1000] Retrieving: NCT00279552
[12/1000] Retrieving: NCT00305097
[13/1000] Retrieving: NCT00312429
[14/1000] Retrieving: NCT00317148
[15/1000] Retrieving: NCT00319748
[16/1000] Retrieving: NCT00324363
[17/1000] Retrieving: NCT00343668
[18/1000] Retrieving: NCT00344318
[19/1000] Retrieving: NCT00351611
[20/1000] Retrieving: NCT00356148
[21/1000] Retrieving: NCT00379366
[22/1000] Retrieving: NCT00391690
[23/1000] Retrieving: NCT00396734
[24/1000] Retrieving: NCT00397215
[25/1000] Retrieving: NCT00401245
[26/1000] Retrieving: NCT00404495
[27/1000] Retrieving: NCT00425789
[28/1000] Retrieving: NCT00426751
[29/1000] Retrieving: NCT00440245
[30/1000] Retrieving: N

In [11]:
print(
    f"Total CHIA rows: "
    f"{len(chia_retrieval_df)}"
)

print(
    f"Successful retrievals: "
    f"{chia_retrieval_df['request_success'].sum()}"
)

print(
    f"Failed retrievals: "
    f"{(~chia_retrieval_df['request_success']).sum()}"
)

print(
    "Missing eligibility text among successful retrievals:",
    (
        chia_retrieval_df.loc[
            chia_retrieval_df["request_success"],
            "eligibility_text",
        ]
        .isna()
        .sum()
    ),
)

print(
    f"Unique benchmark NCT IDs: "
    f"{chia_retrieval_df['nct_id'].nunique()}"
)

print(
    f"Unique resolved NCT IDs: "
    f"{chia_retrieval_df['resolved_nct_id'].nunique()}"
)

print(
    f"Remapped NCT IDs: "
    f"{chia_retrieval_df['id_remapped'].sum()}"
)

Total CHIA rows: 1000
Successful retrievals: 1000
Failed retrievals: 0
Missing eligibility text among successful retrievals: 0
Unique benchmark NCT IDs: 1000
Unique resolved NCT IDs: 1000
Remapped NCT IDs: 1


In [12]:
chia_retrieval_df.loc[
    chia_retrieval_df["id_remapped"],
    [
        "nct_id",
        "resolved_nct_id",
        "id_remapped",
        "request_success",
    ],
]

,nct_id,resolved_nct_id,id_remapped,request_success
591,NCT02822001,NCT02728726,True,True


In [13]:
lct_retrieval_df = retrieve_benchmark_dataset(
    nct_ids=lct_nct_ids,
    dataset_label="LCT",
    output_file=LCT_OUTPUT_FILE,
)

[1/1006] Retrieving: NCT03860012
[2/1006] Retrieving: NCT03860025
[3/1006] Retrieving: NCT03860038
[4/1006] Retrieving: NCT03860064
[5/1006] Retrieving: NCT03860090
[6/1006] Retrieving: NCT03860103
[7/1006] Retrieving: NCT03860116
[8/1006] Retrieving: NCT03860142
[9/1006] Retrieving: NCT03860168
[10/1006] Retrieving: NCT03860181
[11/1006] Retrieving: NCT03860194
[12/1006] Retrieving: NCT03860220
[13/1006] Retrieving: NCT03860233
[14/1006] Retrieving: NCT03860246
[15/1006] Retrieving: NCT03860259
[16/1006] Retrieving: NCT03860311
[17/1006] Retrieving: NCT03860324
[18/1006] Retrieving: NCT03860337
[19/1006] Retrieving: NCT03860350
[20/1006] Retrieving: NCT03860363
[21/1006] Retrieving: NCT03860376
[22/1006] Retrieving: NCT03860389
[23/1006] Retrieving: NCT03860402
[24/1006] Retrieving: NCT03860415
[25/1006] Retrieving: NCT03860428
[26/1006] Retrieving: NCT03860454
[27/1006] Retrieving: NCT03860467
[28/1006] Retrieving: NCT03860480
[29/1006] Retrieving: NCT03860493
[30/1006] Retrieving: N

In [14]:
print(
    f"Total LCT rows: "
    f"{len(lct_retrieval_df)}"
)

print(
    f"Successful retrievals: "
    f"{lct_retrieval_df['request_success'].sum()}"
)

print(
    f"Failed retrievals: "
    f"{(~lct_retrieval_df['request_success']).sum()}"
)

print(
    "Missing eligibility text among successful retrievals:",
    (
        lct_retrieval_df.loc[
            lct_retrieval_df["request_success"],
            "eligibility_text",
        ]
        .isna()
        .sum()
    ),
)

print(
    f"Unique benchmark NCT IDs: "
    f"{lct_retrieval_df['nct_id'].nunique()}"
)

print(
    f"Unique resolved NCT IDs: "
    f"{lct_retrieval_df['resolved_nct_id'].nunique()}"
)

print(
    f"Remapped NCT IDs: "
    f"{lct_retrieval_df['id_remapped'].sum()}"
)

Total LCT rows: 1006
Successful retrievals: 1006
Failed retrievals: 0
Missing eligibility text among successful retrievals: 0
Unique benchmark NCT IDs: 1006
Unique resolved NCT IDs: 1006
Remapped NCT IDs: 0
